# XGBoost Model

## 1. Imports

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import sys
sys.path.append('../')
from src.utils import save_results

## 2. Load Data

In [2]:
print("XGBoost: Loading final pre-processed dataset...")
input_path = Path("../data/processed/final_ml_ready_dataset.csv")
results_path = "../results/model_comparison.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{input_path}'. Please run all data preparation scripts first.")

XGBoost: Loading final pre-processed dataset...
Dataset loaded successfully. Shape: (2619, 202)


## 3. Define Features (X) and Target (y)

In [3]:
target_column = 'is_fraud'
X = df.drop(columns=[target_column])
y = df[target_column]

## 4. Split Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Define and Train Model

In [5]:
print("XGBoost: Training model...")
model_name = "XGBoost"
# XGBoost handles class imbalance with scale_pos_weight
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
hyperparams = {'random_state': 42, 'eval_metric': 'logloss', 'scale_pos_weight': scale_pos_weight, 'n_estimators': 100}
model = xgb.XGBClassifier(**hyperparams)

model.fit(X_train, y_train)
print("Model trained.")

XGBoost: Training model...
Model trained.


## 6. Evaluate Model

In [6]:
print("XGBoost: Evaluating model...")
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

XGBoost: Evaluating model...


## 7. Save Results

In [7]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba)
}

description = f"A gradient boosting framework. Hyperparameters: {hyperparams}"

save_results(results_path, model_name, description, metrics)

Updated results for 'XGBoost' in '..\results\model_comparison.csv'.


## LLM Summary
### Findings
As expected, XGBoost (Extreme Gradient Boosting) is a top-tier performer on this structured dataset. It achieves the highest F1-score (~0.78) and ROC AUC (~0.97) of all the models, indicating it provides the best overall balance of precision and recall and is the most effective at discriminating between classes. Its ability to handle class imbalance internally via `scale_pos_weight` is a key advantage, allowing it to learn effectively from the minority fraud class.
### Insights
For a business seeking the highest performance from a single model, XGBoost is the clear choice. Its high F1-score means it delivers the best blend of catching fraud and avoiding false positives, maximizing the efficiency of a fraud detection pipeline. Its deployment would lead to the highest direct impact on reducing fraud losses while maintaining a good customer experience. The model's complexity is higher than a simple tree or linear model, but its performance justifies the investment.
### Feature Importance Interpretation
XGBoost, like Random Forest, can provide robust feature importance scores. However, it builds trees sequentially, with each new tree correcting the errors of the previous one. This means its feature importance often highlights features that are good at correcting residual errors. We would still expect the same set of top features, but their ranking might differ slightly from Random Forest, potentially giving more weight to features that capture subtle, non-linear interactions that other models missed.